#### Step 1: Import Libraries & API Keys

In [1]:
import os
import groq
import json
import requests
import gradio as gr
from openai import OpenAI
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")


c:\Users\Orchid X\OneDrive\Desktop\AI_Engineering\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Step 2: Set up Pushover

In [ ]:
# a -> Set up account in your browser
# b -> Set up the app on your iPhone/Android phone, log into the same account
# c -> In the browser create an "Application/API token"
# d -> Copy your user key and API token into the .env file
# Like this but without the hashtag symbols and with your own keys:
# PUSHOVER_USER=xxxxxxxxxx
# PUSHOVER_TOKEN=yyyyyyyyy

# Save changes in the .env file

In [2]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [3]:
import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)
    

In [4]:
send_notification("Hello to myself, from this amazing AI Engineering training.")

#### Step 3: Describe Pushover as an LLM tool

In [5]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. Use this to alert the user about important events, completed tasks, or time-sensitive information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message":{
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

#### Step 4: Add Pushover to the list of tools for the LLM

In [6]:
tools = [{"type": "function", "function": send_notification_function}]

#### Step 5: Calling the tool from an LLM

In [7]:
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role":"user", "content": "Please send me a notification telling me what amazing progress\
                I'm making on the AI Engineering training by SuperDataScience."
        }
    ],
    tools=tools,
    tool_choice="auto"
)

# Check if model wants to call a tool

message = response.choices[0].message
print(message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_4xBGdz73Ijaua4EyPX5Pfur4', function=Function(arguments='{"message":"You\'re making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work and stay motivated!"}', name='send_notification'), type='function')])


In [9]:
print(message.tool_calls[0])

ChatCompletionMessageFunctionToolCall(id='call_4xBGdz73Ijaua4EyPX5Pfur4', function=Function(arguments='{"message":"You\'re making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work and stay motivated!"}', name='send_notification'), type='function')


In [10]:
if message.tool_calls:
    tool_call = message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    
    # Actually send the notification
    send_notification(args["message"])
    print(f"Sent notification: {args["message"]}")
else:
    print(message.content)

Sent notification: You're making amazing progress on the AI Engineering training by SuperDataScience! Keep up the great work and stay motivated!
